## 第 3 课：load → compute → store 与元素级运算

题目：[Triton: Element-wise ReLU](https://www.deep-ml.com/problems/969?from=Triton%20Essentials)（ID 969）

计算目标：

In [ ]:
output[i] = max(0, x[i])

例如：

In [ ]:
x = [-2.0, 0.5, 3.0, -1.0]

output = [0.0, 0.5, 3.0, 0.0]

和 Vector Add 相比：输入从两个变成一个，compute 从 `+` 变成 `max(0, x)`。其余一切（offsets、mask、grid）都不变。

### 1. 核心思路不变

开场白（program_id → offsets → mask）和 Vector Add 一模一样，只是少了一个输入：

In [ ]:
pid = tl.program_id(0)
offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
mask = offsets < n_elements

x_block = tl.load(x_ptr + offsets, mask=mask)

### 2. 表达 ReLU 的两种方式

In [ ]:
result = tl.maximum(x_block, 0.0)            # 方式一：逐元素 max
result = tl.where(x_block > 0, x_block, 0.0) # 方式二：条件选择

两者语义相同，在大多数 GPU 上编译出的指令也相同。

注意：这里**不能用** Python 内置的 `max(x_block, 0)` —— Python 的 `max` 是标量运算，而 Triton block 没有 Python 语义。

### 3. 单输入 kernel 的特点

- 只有 `x_ptr` 一个输入指针 + `output_ptr` 一个输出指针。
- 计算是纯元素级的：没有跨 lane 的数据流动（没有归约、没有 shuffle），所以不需要关心 block 内元素的排列方式。

## 你的代码骨架

In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def relu_kernel(
    x_ptr,
    output_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr,
):
    # TODO 1：program ID / offsets / mask（和 Vector Add 一样的开场）

    # TODO 2：加载 x

    # TODO 3：逐元素 ReLU（tl.maximum 或 tl.where）

    # TODO 4：写回 output
    pass


def relu(x: torch.Tensor) -> torch.Tensor:
    BLOCK_SIZE = 1024

    # TODO 5：分配 output（形状、dtype 与 x 相同）

    # TODO 6：创建一维 grid

    # TODO 7：启动 kernel

    # TODO 8：返回 output
    pass

同时回答：

1. 为什么不能用 Python 内置 `max(x_block, 0)` 实现逐元素 ReLU？
2. `tl.maximum(x, 0.0)` 和 `tl.where(x > 0, x, 0.0)` 有什么区别？为什么说它们等价？
3. 如果输入是 `float16`，输出是什么 dtype？`0.0` 这个常量需要手动匹配 dtype 吗？

把代码和三个答案发给我，我继续审查。